In [1]:
!git clone https://github.com/johnma2006/mamba-minimal.git
!pip install transformers datasets accelerate einops

Cloning into 'mamba-minimal'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 57 (delta 29), reused 22 (delta 22), pack-reused 24 (from 1)
Receiving objects: 100% (57/57), 20.35 KiB | 3.39 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [2]:
import sys
sys.path.append('/content/mamba-minimal')

In [4]:
import os
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import sys
sys.path.append('/kaggle/working/mamba-minimal')

from model import Mamba

In [5]:
from model import Mamba

print("Import OK ")

Import OK 


In [10]:
MAX_LEN = 512

BATCH_SIZE = 16   # tăng từ 8 → 16 (an toàn cho T4)

GRAD_ACCUM = 2    # giả lập batch = 32

EPOCHS = 5
LR = 1e-4

D_MODEL = 256
N_LAYER = 4
VOCAB_SIZE = 50257

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [11]:
from datasets import load_dataset

# sửa đúng dataset của bạn ở đây
# ví dụ:
dataset = load_dataset('ag_news')

train_dataset = dataset['train']

test_dataset = dataset['test']

NUM_CLASSES = len(set(train_dataset['label']))

print(train_dataset[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [14]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token

In [17]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN
    )

In [18]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=['text']
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=['text']
)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [19]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,            # ✅ FIX quan trọng
    pin_memory=True,
    persistent_workers=False, # ✅ phải tắt
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,            # ✅ FIX
    pin_memory=True,
    persistent_workers=False, # ✅ FIX
    collate_fn=data_collator
)

In [20]:
class ModelArgs:

    def __init__(self):

        self.d_model = D_MODEL

        self.n_layer = N_LAYER

        self.vocab_size = VOCAB_SIZE

        self.d_state = 16

        self.expand = 2

        # FIX
        self.dt_rank = 16

        self.d_conv = 4

        self.pad_vocab_size_multiple = 8

        self.conv_bias = True

        self.bias = False

        self.d_inner = int(
            self.expand * self.d_model
        )


args = ModelArgs()

In [21]:
class MambaClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.mamba = Mamba(args)

        # 🔥 thêm projection từ vocab → D_MODEL
        self.proj = nn.Linear(50257, D_MODEL)

        self.norm = nn.LayerNorm(D_MODEL)

        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Linear(
            D_MODEL,
            NUM_CLASSES
        )

    def forward(self, input_ids, attention_mask=None):

        # [B, L, VOCAB]
        x = self.mamba(input_ids)

        # lấy token cuối → [B, VOCAB]
        x = x[:, -1, :]

        # map về hidden dim → [B, D_MODEL]
        x = self.proj(x)

        x = self.norm(x)

        x = self.dropout(x)

        x = self.classifier(x)

        return x


model = MambaClassifier().to(DEVICE)

In [22]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()

In [23]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

EPOCHS = 5   # ✅ chạy 3 epoch

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    start = time.time()

    optimizer.zero_grad(set_to_none=True)

    print(f'\nEpoch {epoch+1}/{EPOCHS}')

    for step, batch in enumerate(tqdm(train_loader)):

        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):

            outputs = model(input_ids)

            loss = criterion(outputs, labels)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        # ✅ update khi đủ accumulation hoặc batch cuối
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    end = time.time()

    acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(train_loader)

    print(f'\nLoss: {avg_loss:.4f}')
    print(f'Accuracy: {acc:.4f}')
    print(f'Time: {end-start:.2f} sec')


Epoch 1/5


100%|██████████| 7500/7500 [1:29:12<00:00,  1.40it/s]  



Loss: 1.1342
Accuracy: 0.4323
Time: 5352.67 sec

Epoch 2/5


100%|██████████| 7500/7500 [1:29:40<00:00,  1.39it/s]  



Loss: 0.3582
Accuracy: 0.8743
Time: 5380.27 sec

Epoch 3/5


...


Loss: 0.2342
Accuracy: 0.9226
Time: 5342.82 sec

Epoch 4/5


100%|██████████| 7500/7500 [1:29:05<00:00,  1.40it/s]  



Loss: 0.1637
Accuracy: 0.9493
Time: 5345.82 sec

Epoch 5/5


100%|██████████| 7500/7500 [1:29:17<00:00,  1.40it/s]  


Loss: 0.1188
Accuracy: 0.9656
Time: 5357.48 sec


In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):

        input_ids = batch['input_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())